# H-1B LCA data — exploration

**Purpose:** find every way this data is broken *before* writing `src/clean.py`.

The output of this notebook is the specification for Step 4. Nothing here is
analysis for its own sake — each section answers a question whose answer
changes how the cleaning code has to work.

Source: nine quarterly LCA disclosure files from the DOL Office of Foreign
Labor Certification, covering October 2023 – March 2026.

## Setup

Reading 850 MB of `.xlsx` takes about 15 minutes. The cell below converts each
file to Parquet once and reuses the cache afterwards, which brings a full
reload down to a few seconds.

Two things are already baked in here because Step 2 established them:

- **Sheets are selected by index, not name.** All nine files use different
  sheet names.
- **Blank rows are dropped on read.** 73% of all rows in these sheets are
  empty padding.

Both of these move to `src/ingest.py` in Step 4.

In [1]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, INTERIM = ROOT / "data" / "raw", ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

def read_xlsx(path):
    """Stream one .xlsx to a DataFrame, dropping blank padding rows."""
    wb = load_workbook(path, read_only=True)
    ws = wb[wb.sheetnames[0]]                 # by index - names are inconsistent
    rows = ws.iter_rows(values_only=True)
    hdr = list(next(rows))
    ci = hdr.index("CASE_NUMBER")
    data = [r for r in rows if r[ci] is not None]
    wb.close()
    return pd.DataFrame(data, columns=hdr)

def cached(path):
    dest = INTERIM / (path.stem + ".parquet")
    if not dest.exists():
        df = read_xlsx(path)
        df = df.astype({c: "string" for c in df.columns if df[c].dtype == object})
        df.to_parquet(dest, compression="snappy", index=False)
    return dest

sources = sorted(RAW.glob("*.xlsx"))          # glob: DOL misspelled one filename
print(f"{len(sources)} source files")
for p in sources:
    print(" ", p.name)

9 source files
  LCA_Disclosure_Data_FY2024_Q1.xlsx
  LCA_Disclosure_Data_FY2024_Q2.xlsx
  LCA_Disclosure_Data_FY2024_Q3.xlsx
  LCA_Disclosure_Data_FY2024_Q4.xlsx
  LCA_Disclosure_Data_FY2025_Q1.xlsx
  LCA_Disclosure_Data_FY2025_Q2.xlsx
  LCA_Disclosure_Data_FY2025_Q3.xlsx
  LCA_Disclosure_Data_FY2025_Q4.xlsx
  LCA_Dislclosure_Data_FY2026_Q2.xlsx


## 1. Do the files share a schema?

If columns differ between files, every downstream assumption is unsafe.

In [2]:
info = []
for p in sources:
    df = pd.read_parquet(cached(p))
    info.append({"file": p.name, "rows": len(df), "cols": df.shape[1],
                 "columns": set(df.columns)})

summary = pd.DataFrame([{k: v for k, v in d.items() if k != "columns"} for d in info])
display(summary)

base = info[0]["columns"]
for d in info[1:]:
    extra, missing = d["columns"] - base, base - d["columns"]
    if extra or missing:
        print(f"{d['file']}:  added {sorted(extra)}  removed {sorted(missing)}")

,file,rows,cols
0,LCA_Disclosure_Data_FY2024_Q1.xlsx,99692,97
1,LCA_Disclosure_Data_FY2024_Q2.xlsx,123978,97
2,LCA_Disclosure_Data_FY2024_Q3.xlsx,216470,97
3,LCA_Disclosure_Data_FY2024_Q4.xlsx,120897,97
4,LCA_Disclosure_Data_FY2025_Q1.xlsx,107414,97
5,LCA_Disclosure_Data_FY2025_Q2.xlsx,132133,98
6,LCA_Disclosure_Data_FY2025_Q3.xlsx,238425,98
7,LCA_Disclosure_Data_FY2025_Q4.xlsx,118580,98
8,LCA_Dislclosure_Data_FY2026_Q2.xlsx,210387,98


LCA_Disclosure_Data_FY2025_Q1.xlsx:  added ['H-1B_DEPENDENT']  removed ['H_1B_DEPENDENT']
LCA_Disclosure_Data_FY2025_Q2.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Disclosure_Data_FY2025_Q3.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Disclosure_Data_FY2025_Q4.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Dislclosure_Data_FY2026_Q2.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []


**Finding.** The column set changes **between Q1 and Q2 of FY2025** — not at a
fiscal year boundary. `LAWFIRM_BUSINESS_FEIN` appears and stays.

Consequence for Step 4: do not key schema handling off the year in the
filename. Union the columns and let the missing one be null.

## 2. Load everything and deduplicate

In [3]:
COLS = ["CASE_NUMBER","CASE_STATUS","VISA_CLASS","DECISION_DATE","EMPLOYER_NAME",
        "JOB_TITLE","SOC_CODE","SOC_TITLE","WAGE_RATE_OF_PAY_FROM","WAGE_RATE_OF_PAY_TO",
        "WAGE_UNIT_OF_PAY","PREVAILING_WAGE","PW_UNIT_OF_PAY","WORKSITE_CITY",
        "WORKSITE_STATE","FULL_TIME_POSITION"]

raw = pd.concat([pd.read_parquet(cached(p), columns=COLS) for p in sources],
                ignore_index=True)
df = raw.drop_duplicates(subset="CASE_NUMBER", keep="last")

print(f"rows across all files : {len(raw):,}")
print(f"unique case numbers   : {len(df):,}")
print(f"duplicates removed    : {len(raw) - len(df):,}")

rows across all files : 1,367,976
unique case numbers   : 1,347,103
duplicates removed    : 20,873


**Finding.** 20,873 cases appear in two files each. Every duplicate spans a
quarter boundary and none repeat *within* a file — a case decided near the
cutoff gets published in both quarters.

Consequence: deduplicate on `CASE_NUMBER`. Keeping `last` prefers the more
recent publication.

## 3. Wage units — the single most important column

Wages are meaningless until they are on one scale.

In [4]:
units = df["WAGE_UNIT_OF_PAY"].value_counts(dropna=False)
display(pd.DataFrame({"rows": units, "share": (units / len(df)).map("{:.2%}".format)}))

,rows,share
WAGE_UNIT_OF_PAY,,
Year,1253296,93.04%
Hour,88841,6.59%
Month,2839,0.21%
Week,1268,0.09%
Bi-Weekly,859,0.06%


Five units, and 7% of rows are *not* annual. Comparing raw
`WAGE_RATE_OF_PAY_FROM` across rows would silently compare $45/hour against
$120,000/year.

In [5]:
MULT = {"Year": 1, "Hour": 2080, "Month": 12, "Week": 52, "Bi-Weekly": 26}
wage = pd.to_numeric(df["WAGE_RATE_OF_PAY_FROM"], errors="coerce")
df = df.assign(annual=wage * df["WAGE_UNIT_OF_PAY"].map(MULT))

print("nulls / unparseable :", int(wage.isna().sum()))
print("exactly zero        :", int((wage == 0).sum()))
print("negative            :", int((wage < 0).sum()))
print()
display(df["annual"].describe(percentiles=[.01, .25, .5, .75, .99])
          .apply(lambda x: f"{x:,.0f}").to_frame("annualized USD"))

nulls / unparseable : 0
exactly zero        : 0
negative            : 0



,annualized USD
count,"1,347,103"
mean,"428,938"
std,"9,386,632"
min,"15,080"
1%,"48,443"
25%,"90,002"
50%,"118,248"
75%,"155,605"
99%,"354,679"
max,"1,466,400,000"


**Finding.** No nulls, no zeros, no negatives — better than the plan assumed.

But the mean is **$428,938** against a median of **$118,248**, with a standard
deviation of $9.4M. Something is badly wrong at the top end.

## 4. Where the extreme values come from

In [6]:
worst = df.nlargest(5, "annual")[["annual","WAGE_RATE_OF_PAY_FROM","WAGE_UNIT_OF_PAY",
                                  "JOB_TITLE","EMPLOYER_NAME"]].copy()
worst["annual"] = worst["annual"].map("${:,.0f}".format)
display(worst.reset_index(drop=True))

,annual,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,JOB_TITLE,EMPLOYER_NAME
0,"$1,466,400,000",705000.0,Hour,"=""Physician (Interventional Cardiologist)""",Ascension Medical Group ProMed
1,"$1,000,731,680",481121.0,Hour,Physician,Allegheny Clinic
2,"$936,000,000",450000.0,Hour,Hematologist/Oncologist Physician,"North Shore Hematology Oncology Associates, P.C."
3,"$936,000,000",450000.0,Hour,"Vice President, Capital & Partner Solutions","Vista Equity Partners Management, LLC"
4,"$936,000,000",450000.0,Hour,"Assistant Professor, NTT, Clinical",University of Texas Medical Branch


**Finding — mislabelled units.** Every one of these is marked `Hour` while the
value is plainly an annual salary. $450,000 filed as an hourly rate becomes
$936,000,000 after multiplying by 2080.

This is not a rare typo:

In [7]:
hourly = df[df["WAGE_UNIT_OF_PAY"] == "Hour"].copy()
hourly["rate"] = pd.to_numeric(hourly["WAGE_RATE_OF_PAY_FROM"], errors="coerce")
suspect = hourly[hourly["rate"] > 500]

print(f"hourly rows                 : {len(hourly):,}")
print(f"  ... with rate > $500/hr   : {len(suspect):,}  ({len(suspect)/len(hourly):.2%})")
print(f"  their median raw value    : ${suspect['rate'].median():,.0f}  <- an annual salary")
print(f"  genuine hourly median     : ${hourly[hourly['rate'] <= 500]['rate'].median():,.2f}/hr")

hourly rows                 : 88,841
  ... with rate > $500/hr   : 1,618  (1.82%)
  their median raw value    : $105,227  <- an annual salary
  genuine hourly median     : $45.00/hr


In [8]:
LO, HI = 10_000, 2_000_000
print(f"below ${LO:,}    : {int((df['annual'] < LO).sum()):,}")
print(f"above ${HI:,}  : {int((df['annual'] > HI).sum()):,}"
      f"  ({(df['annual'] > HI).mean():.3%})")
print(f"inside band     : {int(df['annual'].between(LO, HI).sum()):,}"
      f"  ({df['annual'].between(LO, HI).mean():.3%})")

below $10,000    : 0
above $2,000,000  : 3,270  (0.243%)
inside band     : 1,343,833  (99.757%)


**Finding.** The plan's $10k floor never fires — the true minimum is $15,080
(the federal minimum wage annualized, which is legitimate). The $2M ceiling
catches 3,270 rows, 0.24%.

Consequence: flag rather than delete, and treat `Hour` rows above ~$500 as
mislabelled annual figures rather than discarding them.

## 5. Excel escape artifacts

The quietest bug in this dataset.

In [9]:
for col in ["JOB_TITLE", "SOC_CODE", "EMPLOYER_NAME", "WORKSITE_CITY", "SOC_TITLE"]:
    s = df[col].astype("string")
    n = int(s.str.startswith('="', na=False).sum())
    print(f"{col:<16} {n:>8,}  ({n/len(df):.3%})")

print()
for v in df.loc[df["JOB_TITLE"].astype("string").str.startswith('="', na=False),
                "JOB_TITLE"].head(3):
    print(" ", repr(v))

JOB_TITLE         130,298  (9.672%)


SOC_CODE          130,282  (9.671%)


EMPLOYER_NAME           0  (0.000%)


WORKSITE_CITY           0  (0.000%)


SOC_TITLE               0  (0.000%)



  '="Corporate Steel Analyst ("Steel Analyst, Metallurgist")"'
  '="Early Learning Center ("ELC") Fellow"'
  '="Financial Planning and Analysis ("FP&A") Manager"'


Values whose text contains a double quote were exported wrapped in an Excel
formula escape: `="Financial Planning and Analysis ("FP&A") Manager"`.

It affects `JOB_TITLE` and `SOC_CODE`. Here is what it costs:

In [10]:
soc = df["SOC_CODE"].astype("string").str.strip()
soc_clean = soc.str.replace(r'^="|"$', "", regex=True).str.strip()

def is_tech(series):
    return series.str.slice(0, 2).eq("15") | series.str.startswith("11-3021", na=False)

naive, fixed = is_tech(soc), is_tech(soc_clean)
print(f"tech rows, filtering raw SOC_CODE   : {int(naive.sum()):,}")
print(f"tech rows, after stripping the =\"   : {int(fixed.sum()):,}")
print(f"silently dropped                    : {int(fixed.sum() - naive.sum()):,}"
      f"  ({(fixed.sum() - naive.sum()) / fixed.sum():.1%} of all tech filings)")

tech rows, filtering raw SOC_CODE   : 784,167
tech rows, after stripping the ="   : 866,685
silently dropped                    : 82,518  (9.5% of all tech filings)


**Finding.** A naive tech filter loses **82,518 filings — 9.5% of the tech
data** — and fails silently. No error, no warning, just a smaller number.

This is the single most valuable thing in this notebook. Strip the escape
before any string comparison.

## 6. Employer name fragmentation

In [11]:
emp = df["EMPLOYER_NAME"].astype("string")
norm = (emp.str.upper()
           .str.replace(r'[^\w\s]', "", regex=True)
           .str.replace(r'\s+', " ", regex=True).str.strip()
           .str.replace(r'\s+(INC|LLC|LTD|CORP|CORPORATION|CO|LP|LLP|PC|PLLC)$',
                        "", regex=True))

print(f"distinct raw names   : {emp.nunique():,}")
print(f"after normalization  : {norm.nunique():,}")
print(f"collapsed            : {emp.nunique() - norm.nunique():,}"
      f"  ({1 - norm.nunique()/emp.nunique():.1%})")
print()
for v in sorted(emp[emp.str.contains("COGNIZANT", case=False, na=False)].unique())[:6]:
    print(" ", repr(v))

distinct raw names   : 120,779
after normalization  : 102,660


collapsed            : 18,119  (15.0%)



  'COGNIZANT TECHNOLOGY SOLUTIONS US CORP'
  'COGNIZANT WORLDWIDE LIMITED'
  'Cognizant Mobility, Inc.'
  'Cognizant TriZetto Software Group, Inc.'
  'SparkCognizant Inc'
  'TMG HEALTH - A COGNIZANT COMPANY'


**Finding.** Case, punctuation, and corporate suffixes collapse 15% of
distinct names.

**But look at the last two examples.** `SparkCognizant Inc` and
`TMG HEALTH - A COGNIZANT COMPANY` are different companies that merely share a
substring. Any fuzzy matching would merge them wrongly.

Consequence: normalize conservatively — case, punctuation, whitespace, and
trailing legal suffixes only. No fuzzy matching in v1, and say so in the
README.

## 7. Occupations and geography

In [12]:
tech = df[fixed.values]
print(f"tech filings: {len(tech):,}  ({len(tech)/len(df):.1%} of all)\n")
display(tech["SOC_TITLE"].value_counts().head(10).to_frame("filings"))

tech filings: 866,685  (64.3% of all)



,filings
SOC_TITLE,
Software Developers,418232
Computer Systems Engineers/Architects,69040
Information Technology Project Managers,47804
Software Quality Assurance Analysts and Testers,44105
Data Scientists,40353
Computer Systems Analysts,35356
Computer Programmers,31845
Computer and Information Systems Managers,30558
Business Intelligence Analysts,28277


In [13]:
city = df["WORKSITE_CITY"].astype("string")
print(f"distinct cities, raw      : {city.nunique():,}")
print(f"after upper + strip       : {city.str.upper().str.strip().nunique():,}")
print(f"distinct states           : {df['WORKSITE_STATE'].nunique()}")
print(f"null city / null state    : {int(city.isna().sum())} / {int(df['WORKSITE_STATE'].isna().sum())}")
print()
top = (tech.groupby([tech['WORKSITE_CITY'].astype('string').str.title(),
                     tech['WORKSITE_STATE']])['CASE_NUMBER'].count()
          .nlargest(8))
display(top.to_frame("filings"))

distinct cities, raw      : 18,879


after upper + strip       : 12,010
distinct states           : 55
null city / null state    : 0 / 0



,,filings
WORKSITE_CITY,WORKSITE_STATE,
New York,NY,35550
Seattle,WA,27801
Austin,TX,22985
Sunnyvale,CA,19756
Plano,TX,19648
Irving,TX,18871
San Francisco,CA,18301
San Jose,CA,18294


**Finding.** Case variation alone accounts for 6,869 phantom cities
(18,879 → 12,010). 55 distinct states — more than 50 because territories are
included. No nulls in either field.

## 8. Case status and visa class

In [14]:
display(df["CASE_STATUS"].value_counts(dropna=False).to_frame("rows"))
display(df["VISA_CLASS"].value_counts(dropna=False).to_frame("rows"))

,rows
CASE_STATUS,
Certified,1236211
Certified - Withdrawn,79588
Withdrawn,21940
Denied,9364


,rows
VISA_CLASS,
H-1B,1312464
E-3 Australian,25310
H-1B1 Chile,5479
H-1B1 Singapore,3850


**Finding.** 7.5% of filings are `Withdrawn` or `Denied` and do not represent
wages anyone committed to pay. About 3% of rows are not H-1B at all — E-3
Australian and H-1B1 Chile/Singapore share the same form.

---

# Data problems found — the spec for Step 4

Every item below was measured above, not assumed. `src/clean.py` must handle
each one.

| # | Problem | Scale | Required handling |
|---|---|---|---|
| 1 | Blank padding rows | 3,610,511 rows (73% of sheets) | Drop where `CASE_NUMBER` is null, on read |
| 2 | Sheet names differ per file | all 9 differ | Select sheet by index, never by name |
| 3 | Column set changes mid-FY2025 | +1 column from FY2025 Q2 | Union columns; do not key off filename year |
| 4 | DOL misspelled a filename | `Dislclosure` | Glob for source files |
| 5 | Duplicate cases across files | 20,873 | Deduplicate on `CASE_NUMBER`, keep last |
| 6 | Mixed wage units | 7% not annual | Annualize: hour x2080, week x52, bi-weekly x26, month x12 |
| 7 | Annual salaries filed as hourly | 1,618 rows | Treat `Hour` with rate > $500 as already annual |
| 8 | Extreme wages | 3,270 above $2M | Flag `is_outlier`, never delete |
| 9 | Excel escape on JOB_TITLE | 130,298 rows (9.7%) | Strip `="` and trailing `"` |
| 10 | Excel escape on SOC_CODE | 130,282 rows | Strip before filtering — **costs 9.5% of tech rows otherwise** |
| 11 | SOC detail suffixes | `15-1252.00` vs `15-1252` | Truncate to 7 characters for grouping |
| 12 | Employer name fragmentation | 15% collapsible | Case, punctuation, suffix only. No fuzzy matching |
| 13 | City case variation | 18,879 → 12,010 | Title-case city, upper-case state |
| 14 | Non-certified filings | 7.5% | Keep `Certified` and `Certified - Withdrawn` only |
| 15 | Non-H-1B visa classes | ~3% | Decide explicitly; document either way |

## What surprised me

- **The plan's biggest predicted risk did not happen.** Column names are stable
  across all nine files apart from one addition, so no alias mapping is needed.
- **A risk nobody predicted did happen.** The `="` escape on `SOC_CODE` would
  have quietly removed 9.5% of the tech data with no error. Naive code would
  have produced plausible-looking, wrong numbers.
- **The wage floor was unnecessary.** No row annualizes below $15,080. The
  damage is all at the top, and it comes from mislabelled units rather than
  from genuinely extreme salaries.